## 3. Modeling TF-IDF x BILSTM

### 3.1 Importing the Libraries 

In [11]:
import os
import numpy as np
import pickle 
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Embedding, Bidirectional, LSTM, Input, Concatenate
from tensorflow.keras.callbacks import EarlyStopping

In [12]:
MODEL_SAVE_PATH = "models/bilstm_model.h5"

### 3.2 Define the Parameter

In [13]:
VOCAB_SIZE = 5000
TFIDF_FEATURES = 5000
MAX_LEN = 200

### 3.2 Load the Dataset

In [14]:
X_train_pad   = pickle.load(open("../dataset/processed/X_train_pad.pkl", "rb"))
X_train_tfidf = pickle.load(open("../dataset/processed/X_train_tfidf.pkl", "rb"))
y_train       = pickle.load(open("../dataset/processed/y_train.pkl", "rb"))

X_test_pad    = pickle.load(open("../dataset/processed/X_test_pad.pkl", "rb"))
X_test_tfidf  = pickle.load(open("../dataset/processed/X_test_tfidf.pkl", "rb"))
y_test        = pickle.load(open("../dataset/processed/y_test.pkl", "rb"))

### 3.3 Check Array Dimension

In [15]:
print(f"Shape X_train_pad: {X_train_pad.shape}")
print(f"Shape X_train_tfidf: {X_train_tfidf.shape}")
print(f"Shape y_train: {y_train.shape}")
print(f"Shape y_test: {y_test.shape}")

Shape X_train_pad: (2526, 200)
Shape X_train_tfidf: (2526, 5000)
Shape y_train: (2526,)
Shape y_test: (632,)


### 3.4 Build BiLSTM Model with TF-IDF

In [16]:
# input squence for bilstm
input_seq = Input(shape=(MAX_LEN,), name='input_sequence')
x = Embedding(input_dim=VOCAB_SIZE, output_dim=128)(input_seq)
x = Bidirectional(LSTM(64))(x)

# input vektor tf-idf
input_tfidf = Input(shape=(TFIDF_FEATURES,), name='input_tfidf')
y = Dense(32, activation='relu')(input_tfidf)

# concate
combined = Concatenate()([x, y])

# output layer
z = Dense(64, activation="relu")(combined)
output_layer = Dense(1, activation="sigmoid")(z)

# define model with two input
model = Model(inputs=[input_seq, input_tfidf], outputs=output_layer)

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
print(model.summary())

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_sequence      │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 200, 128)  │    640,000 │ input_sequence[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_tfidf         │ (None, 5000)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 128)       │     98,816 │ embedding_1[0][0] │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 32)        │    160,032 │ input_tfidf[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 160)       │          0 │ bidirectional_1[… │
│ (Concatenate)       │                   │            │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 64)        │     10,304 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 1)         │         65 │ dense_4[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 909,217 (3.47 MB)

 Trainable params: 909,217 (3.47 MB)

 Non-trainable params: 0 (0.00 B)

None


### 3.5 Define Callbacks

In [17]:
early_stopping = EarlyStopping(
    monitor='val_loss', 
    patience=3, 
    restore_best_weights=True,
    verbose=1
)

### 3.6 Training

In [18]:
history = model.fit(
    [X_train_pad, X_train_tfidf],
    y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 5s 91ms/step - accuracy: 0.8322 - loss: 0.4560 - val_accuracy: 0.9605 - val_loss: 0.1393
Epoch 2/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 3s 77ms/step - accuracy: 0.9757 - loss: 0.0712 - val_accuracy: 0.9704 - val_loss: 0.0857
Epoch 3/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 74ms/step - accuracy: 0.9946 - loss: 0.0167 - val_accuracy: 0.9704 - val_loss: 0.0862
Epoch 4/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 75ms/step - accuracy: 0.9990 - loss: 0.0045 - val_accuracy: 0.9822 - val_loss: 0.0673
Epoch 5/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 74ms/step - accuracy: 1.0000 - loss: 9.1255e-04 - val_accuracy: 0.9862 - val_loss: 0.0704
Epoch 6/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 76ms/step - accuracy: 0.9995 - loss: 0.0016 - val_accuracy: 0.9862 - val_loss: 0.0754
Epoch 7/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 74ms/step - accuracy: 1.0000 - loss: 3.5911e-04 - val_accuracy: 0.9881 - val_loss: 0.0742
Epoch 7: early stopping
Restoring model weights from the end of the best epoch: 4.


### 3.7 Evaluation

In [19]:
loss, acc = model.evaluate([X_test_pad, X_test_tfidf], y_test, verbose=1)
print(f"\nAkurasi Test (BiLSTM + TF-IDF): {acc:.4f}")

20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.9731 - loss: 0.0956

Akurasi Test (BiLSTM + TF-IDF): 0.9731


### 3.8 Save Model

In [20]:
os.makedirs("models", exist_ok=True)
model.save(MODEL_SAVE_PATH)
print("Model saved to:", MODEL_SAVE_PATH)

Model saved to: models/bilstm_model.h5
